In [4]:
from pathlib import Path
import json, numpy as np, pandas as pd, xgboost as xgb

SEED = 42
np.random.seed(SEED)

ROOT      = Path(r"C:\Fintech-Project\graph_AML_pipeline")
DATA_PROC = ROOT / "data" / "processed"
OUT_DIR   = ROOT / "outputs"

# find each artefact anywhere under outputs/, so a models/ subfolder is fine
def find_one(name):
    hits = list(OUT_DIR.rglob(name))
    assert len(hits) == 1, f"{name}: expected 1 match, found {len(hits)} -> {hits}"
    return hits[0]

A = {n: find_one(n) for n in [
    "m1_baseline.json", "m1_test_probs.parquet",
    "06_m1_threshold.json", "06_m1_metrics.csv", "06_m1_features.json",
]}
for k, v in A.items():
    print(f"{k:28s} {v.relative_to(ROOT)}")
print("\nxgboost", xgb.__version__, "| pandas", pd.__version__)

m1_baseline.json             outputs\models\m1_baseline.json
m1_test_probs.parquet        outputs\models\m1_test_probs.parquet
06_m1_threshold.json         outputs\models\06_m1_threshold.json
06_m1_metrics.csv            outputs\tables\06_m1_metrics.csv
06_m1_features.json          outputs\models\06_m1_features.json

xgboost 2.1.4 | pandas 2.2.3


In [5]:
# load the saved m1 model and pull out the settings it still remembers
m1 = xgb.Booster()
m1.load_model(str(A["m1_baseline.json"]))
cfg = json.loads(m1.save_config())

# the config is deeply nested, so search it by key name instead of by path
def dig(obj, key):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == key: return v
            hit = dig(v, key)
            if hit is not None: return hit
    elif isinstance(obj, list):
        for v in obj:
            hit = dig(v, key)
            if hit is not None: return hit
    return None

# only max_depth and scale_pos_weight are reliable here; the other settings
# read back as library defaults whether or not they were used in training
spw = float(dig(cfg, "scale_pos_weight"))
print(f"scale_pos_weight {spw:.5f}   max_depth {dig(cfg, 'max_depth')}   device {dig(cfg, 'device')}")

# read the feature list and the frozen threshold saved by notebook 06
feats = json.load(open(A["06_m1_features.json"]))
thr   = json.load(open(A["06_m1_threshold.json"]))
print(f"\nM1 features ({len(feats)}): {feats}")
print(f"threshold {thr}")

# stop here if anything drifted from the registered plan
assert len(feats) == 10, f"expected 10 M1 features, got {len(feats)}"
assert int(dig(cfg, "max_depth")) == 6
assert abs(spw - 1243.7) < 0.1, f"expected training-window 1243.7, got {spw}"
assert thr["chosen_on"] == "validation"
assert thr["n_trees_used"] == 482 and thr["num_boost_round"] == 500
print("\nchecks passed")

scale_pos_weight 1243.73279   max_depth 6   device cpu

M1 features (10): ['Amount Received', 'Amount Paid', 'From Bank', 'To Bank', 'Receiving Currency', 'Payment Currency', 'Payment Format', 'hour', 'day_of_week', 'is_weekend']
threshold {'threshold': 0.9579981881141687, 'val_f1': 0.12072243346007605, 'chosen_on': 'validation', 'n_trees_used': 482, 'num_boost_round': 500, 'early_stopping_rounds': 50}

checks passed


In [6]:
import json
import numpy as np
import pandas as pd
import xgboost as xgb
from pathlib import Path
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score

SEED = 42
np.random.seed(SEED)

DATA_PROC = Path('../data/processed')
OUT_DIR   = Path('../outputs')

print('xgboost', xgb.__version__)

xgboost 2.1.4


In [7]:
df = pd.read_parquet(DATA_PROC / '05_features_full.parquet')

print(f'rows: {len(df):,}  cols: {df.shape[1]}')
assert len(df) == 5_078_345, 'row count does not match the registered artefact'


rows: 5,078,345  cols: 28


In [8]:
df['hour']        = df['Timestamp'].dt.hour
df['day_of_week'] = df['Timestamp'].dt.dayofweek
df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)

CAT_COLS = ['Receiving Currency', 'Payment Currency', 'Payment Format']
for c in CAT_COLS:
    df[c] = df[c].astype('category').cat.codes

# compare this run's codes against the mapping notebook 06 saved
maps = json.load(open(OUT_DIR / 'models' / '06_category_maps.json'))
for c in CAT_COLS:
    expected = {int(k): v for k, v in maps[c].items()}
    actual = dict(enumerate(pd.read_parquet(DATA_PROC / '05_features_full.parquet',
                                            columns=[c])[c].astype('category').cat.categories))
    assert expected == actual, f'category codes differ from notebook 06 for {c}'
    print(f'{c} -> {len(actual)} categories, codes match')

Receiving Currency -> 15 categories, codes match
Payment Currency -> 15 categories, codes match
Payment Format -> 7 categories, codes match


In [9]:
TAB_FEATS = json.load(open(OUT_DIR / 'models' / '06_m1_features.json'))

GRAPH_FEATS = [
    'from_g_pagerank', 'from_g_in_degree', 'from_g_out_degree',
    'from_g_in_weighted', 'from_g_out_weighted', 'from_g_clustering',
    'from_g_net_flow', 'from_g_two_hop_out',
    'to_g_pagerank', 'to_g_in_degree', 'to_g_out_degree',
    'to_g_in_weighted', 'to_g_out_weighted', 'to_g_clustering',
    'to_g_net_flow', 'to_g_two_hop_out',
]
M2_FEATS = TAB_FEATS + GRAPH_FEATS

missing = [f for f in M2_FEATS if f not in df.columns]
assert not missing, f'columns not in the feature table: {missing}'
assert len(TAB_FEATS) == 10 and len(GRAPH_FEATS) == 16 and len(M2_FEATS) == 26
assert M2_FEATS[:10] == TAB_FEATS, 'the first ten features must stay in m1 order'
assert df[M2_FEATS].isna().sum().sum() == 0, 'unexpected missing values after imputation'

json.dump(M2_FEATS, open(OUT_DIR / 'models' / '07_m2_features.json', 'w'), indent=2)
print(f'M2 uses {len(M2_FEATS)} features, no missing values')

M2 uses 26 features, no missing values


In [10]:
m1p = pd.read_parquet(OUT_DIR / 'models' / 'm1_test_probs.parquet')

test_idx = m1p.index
main_idx = m1p.index[m1p['is_main'].values]

assert test_idx.is_unique
assert test_idx.isin(df.index).all(), 'm1 probabilities reference rows not in this table'
assert len(test_idx) == 761_639 and len(main_idx) == 760_531
assert df.loc[main_idx, 'Is Laundering'].sum() == 906
assert df.loc[test_idx, 'Is Laundering'].sum() == 1_561
print(f'test-full {len(test_idx):,} · test-main {len(main_idx):,} · aligned by index')

test-full 761,639 · test-main 760,531 · aligned by index


In [11]:
train_mask = df['split'] == 'train'
val_mask   = df['split'] == 'val'

X_train, y_train = df.loc[train_mask, M2_FEATS], df.loc[train_mask, 'Is Laundering']
X_val,   y_val   = df.loc[val_mask,   M2_FEATS], df.loc[val_mask,   'Is Laundering']
X_main,  y_main  = df.loc[main_idx,   M2_FEATS], df.loc[main_idx,   'Is Laundering']
X_test,  y_test  = df.loc[test_idx,   M2_FEATS], df.loc[test_idx,   'Is Laundering']

dtrain = xgb.QuantileDMatrix(X_train, label=y_train)
dval   = xgb.QuantileDMatrix(X_val,  label=y_val,  ref=dtrain)
dmain  = xgb.QuantileDMatrix(X_main, label=y_main, ref=dtrain)
dtest  = xgb.QuantileDMatrix(X_test, label=y_test, ref=dtrain)

scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'aucpr',
    'learning_rate': 0.05,
    'max_depth': 6,
    'min_child_weight': 10,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': scale_pos,
    'max_delta_step': 1,
    'tree_method': 'hist',
    'device': 'cpu',
    'seed': SEED,
}

# write the settings to disk so "identical hyperparameters" is an artefact, not a claim
json.dump({k: (float(v) if isinstance(v, (int, float, np.floating)) else v)
           for k, v in params.items()},
          open(OUT_DIR / 'models' / '07_params.json', 'w'), indent=2)

assert abs(scale_pos - 1243.7) < 0.1, f'scale_pos_weight off-spec: {scale_pos}'
print(f'scale_pos_weight = {scale_pos:.5f}  ·  train {len(y_train):,} / {int(y_train.sum()):,} illicit')

scale_pos_weight = 1243.73284  ·  train 3,554,957 / 2,856 illicit


In [12]:
booster = xgb.train(
    params, dtrain, num_boost_round=500,
    evals=[(dtrain, 'train'), (dval, 'val')],
    early_stopping_rounds=50,
    verbose_eval=50,
)
print(f'best iteration: {booster.best_iteration}')

[0]	train-aucpr:0.00792	val-aucpr:0.00609
[50]	train-aucpr:0.11043	val-aucpr:0.04937
[100]	train-aucpr:0.14986	val-aucpr:0.05542
[150]	train-aucpr:0.18549	val-aucpr:0.06083
[200]	train-aucpr:0.21762	val-aucpr:0.06585
[250]	train-aucpr:0.24287	val-aucpr:0.07030
[300]	train-aucpr:0.26848	val-aucpr:0.07327
[350]	train-aucpr:0.29108	val-aucpr:0.07563
[400]	train-aucpr:0.31718	val-aucpr:0.07711
[450]	train-aucpr:0.33473	val-aucpr:0.07824
[479]	train-aucpr:0.34897	val-aucpr:0.07875
best iteration: 429


In [13]:
p_val = booster.predict(dval, iteration_range=(0, booster.best_iteration + 1))

grid = np.unique(np.quantile(p_val, np.linspace(0.90, 0.99999, 400)))
f1s  = [f1_score(y_val, (p_val >= t).astype(int), zero_division=0) for t in grid]
best_i = int(np.argmax(f1s))

assert best_i > 0, 'optimum at grid floor — widen the quantile range below 0.90'

THRESH  = float(grid[best_i])
N_TREES = int(booster.best_iteration) + 1
print(f'frozen threshold = {THRESH:.6f}   val minority-F1 = {f1s[best_i]:.4f}   trees = {N_TREES}')

json.dump({'threshold': THRESH, 'val_f1': float(f1s[best_i]),
           'chosen_on': 'validation', 'n_trees_used': N_TREES,
           'num_boost_round': 500, 'early_stopping_rounds': 50},
          open(OUT_DIR / 'models' / '07_m2_threshold.json', 'w'), indent=2)

frozen threshold = 0.953385   val minority-F1 = 0.1521   trees = 430


In [14]:
def evaluate(bst, dm, y, label, thresh):
    p = bst.predict(dm, iteration_range=(0, bst.best_iteration + 1))
    yhat = (p >= thresh).astype(int)
    return {
        'population': label,
        'n': int(len(y)),
        'illicit': int(y.sum()),
        'pr_auc': float(average_precision_score(y, p)),
        'f1_minority': float(f1_score(y, yhat, zero_division=0)),
        'roc_auc': float(roc_auc_score(y, p)),
    }, p

m_main, p_main = evaluate(booster, dmain, y_main, 'test-main', THRESH)
m_full, p_test = evaluate(booster, dtest, y_test, 'test-full', THRESH)
results = pd.DataFrame([m_main, m_full])
print(results.to_string(index=False))

booster.save_model(OUT_DIR / 'models' / 'm2_graph_xgb.json')
pd.DataFrame({'m2_prob': p_test, 'is_main': m1p['is_main'].values}, index=test_idx) \
  .to_parquet(OUT_DIR / 'models' / 'm2_test_probs.parquet')
results.to_csv(OUT_DIR / 'tables' / '07_m2_metrics.csv', index=False)

assert m_main['n'] == 760_531 and m_full['n'] == 761_639, 'population sizes off-spec'

if m_main['pr_auc'] > 0.90:
    print('\nSTOP — PR-AUC above 0.90. Run the label-permutation test before continuing.')

population      n  illicit   pr_auc  f1_minority  roc_auc
 test-main 760531      906 0.063708     0.119140 0.959068
 test-full 761639     1561 0.114237     0.176979 0.961651


In [15]:
m2p = pd.read_parquet(OUT_DIR / 'models' / 'm2_test_probs.parquet')

# pair the two models row by row on the index, restricted to test-main
assert m1p.index.equals(m2p.index), 'probability files are not on the same index'
p1 = m1p.loc[main_idx, 'm1_prob'].to_numpy()
p2 = m2p.loc[main_idx, 'm2_prob'].to_numpy()
y  = df.loc[main_idx, 'Is Laundering'].to_numpy()

# each model keeps the threshold frozen on its own validation split
T1 = json.load(open(OUT_DIR / 'models' / '06_m1_threshold.json'))['threshold']
T2 = json.load(open(OUT_DIR / 'models' / '07_m2_threshold.json'))['threshold']

point = {
    'pr_auc_diff':  average_precision_score(y, p2) - average_precision_score(y, p1),
    'f1_diff':      f1_score(y, p2 >= T2, zero_division=0) - f1_score(y, p1 >= T1, zero_division=0),
    'roc_auc_diff': roc_auc_score(y, p2) - roc_auc_score(y, p1),
}
print('point estimates (M2 - M1):', {k: round(v, 6) for k, v in point.items()})

# resample rows, not models, so the shared test-set variance cancels
n, B = len(y), 1000
rng = np.random.default_rng(SEED)
d_pr, d_f1, d_roc = [], [], []
for b in range(B):
    idx = rng.integers(0, n, n)
    yb = y[idx]
    if yb.sum() < 1:
        continue
    p1b, p2b = p1[idx], p2[idx]
    d_pr.append(average_precision_score(yb, p2b) - average_precision_score(yb, p1b))
    d_f1.append(f1_score(yb, p2b >= T2, zero_division=0) - f1_score(yb, p1b >= T1, zero_division=0))
    d_roc.append(roc_auc_score(yb, p2b) - roc_auc_score(yb, p1b))
    if (b + 1) % 100 == 0:
        print(f'  {b + 1}/{B}')

def ci(v):
    return [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))]

out = {
    'n_test_main': n, 'illicit': int(y.sum()), 'B': len(d_pr),
    'thresh_m1': T1, 'thresh_m2': T2,
    'pr_auc_diff_point': point['pr_auc_diff'], 'pr_auc_diff_ci': ci(d_pr),
    'f1_diff_point': point['f1_diff'],         'f1_diff_ci': ci(d_f1),
    'roc_auc_diff_point': point['roc_auc_diff'], 'roc_auc_diff_ci': ci(d_roc),
}

# H1 is decided on PR-AUC alone; F1 and ROC-AUC are reported, not deciding
out['h1_supported'] = bool(out['pr_auc_diff_ci'][0] > 0)

for k, v in out.items():
    print(f'{k}: {v}')
pd.DataFrame([out]).to_csv(OUT_DIR / 'tables' / '07_paired_bootstrap.csv', index=False)

point estimates (M2 - M1): {'pr_auc_diff': -0.022058, 'f1_diff': -0.008716, 'roc_auc_diff': -0.005783}
  100/1000
  200/1000
  300/1000
  400/1000
  500/1000
  600/1000
  700/1000
  800/1000
  900/1000
  1000/1000
n_test_main: 760531
illicit: 906
B: 1000
thresh_m1: 0.9579981881141687
thresh_m2: 0.9533845419883794
pr_auc_diff_point: -0.02205818995540823
pr_auc_diff_ci: [-0.0394822499886331, -0.006448303309071923]
f1_diff_point: -0.008716339810992083
f1_diff_ci: [-0.02415565826912572, 0.007021171441187325]
roc_auc_diff_point: -0.005783198038709303
roc_auc_diff_ci: [-0.008886502214629017, -0.0024543387596447697]
h1_supported: False


In [17]:
imp = json.load(open(DATA_PROC / '05_imputation_values.json'))

# decision (b): one median per feature, applied to both sides;
# the three count features are filled with zero
FILL = dict(imp['account_basis_applied'])
for b in ['g_in_degree', 'g_out_degree', 'g_two_hop_out']:
    FILL[b] = imp['count_features_filled_with']
print(f'{len(FILL)} constants:', FILL)

# an account is unseen in training if all eight of its features
# sit exactly on the imputed constants
def imputed_side(frame, prefix):
    hit = pd.Series(True, index=frame.index)
    for b, val in FILL.items():
        hit &= (frame[f'{prefix}{b}'] == val)
    return hit

main = df.loc[main_idx]
from_imp = imputed_side(main, 'from_')
to_imp   = imputed_side(main, 'to_')
n_imp    = from_imp.astype(int) + to_imp.astype(int)

strata = {'both_active': n_imp == 0, 'one_imputed': n_imp == 1, 'both_imputed': n_imp == 2}
for name, m in strata.items():
    print(f'{name:14s} {int(m.sum()):>7,} rows · {int(main.loc[m, "Is Laundering"].sum()):>3} illicit')

assert sum(int(m.sum()) for m in strata.values()) == 760_531

8 constants: {'g_pagerank': 1.3446550120393992e-06, 'g_in_weighted': 12302.345, 'g_out_weighted': 8657.125, 'g_clustering': 0.0, 'g_net_flow': 0.0, 'g_in_degree': 0, 'g_out_degree': 0, 'g_two_hop_out': 0}
both_active    758,982 rows · 878 illicit
one_imputed      1,491 rows ·  28 illicit
both_imputed        58 rows ·   0 illicit


In [19]:
# degeneracy rule: fewer than 1,000 rows or fewer than 10 illicit means
# counts only, no performance claim
rows = []
for name, m in strata.items():
    mm = m.to_numpy()
    yb, p1b, p2b = y[mm], p1[mm], p2[mm]
    r = {'stratum': name, 'n': int(mm.sum()), 'illicit': int(yb.sum())}
    if r['n'] < 1000 or r['illicit'] < 10:
        r.update({'m1_pr_auc': None, 'm2_pr_auc': None, 'diff': None,
                  'note': 'degenerate - counts only'})
    else:
        a1 = average_precision_score(yb, p1b)
        a2 = average_precision_score(yb, p2b)
        r.update({'m1_pr_auc': round(a1, 6), 'm2_pr_auc': round(a2, 6),
                  'diff': round(a2 - a1, 6), 'note': ''})
    rows.append(r)

strata_tbl = pd.DataFrame(rows)
print(strata_tbl.to_string(index=False))
strata_tbl.to_csv(OUT_DIR / 'tables' / '07_staleness_strata.csv', index=False)

     stratum      n  illicit  m1_pr_auc  m2_pr_auc      diff                     note
 both_active 758982      878   0.086591   0.064119 -0.022472                         
 one_imputed   1491       28   0.101299   0.073938 -0.027361                         
both_imputed     58        0        NaN        NaN       NaN degenerate - counts only


In [20]:
# supplementary, not confirmatory. pre-specified 7 Sep 2026 in
# docs/DECISION_2026-09-07_round_budget.md, before M2 was fitted.
# new variable name so the registered 429-iteration booster stays alive.
booster_supp = xgb.train(
    params, dtrain, num_boost_round=2000,
    evals=[(dtrain, 'train'), (dval, 'val')],
    early_stopping_rounds=50,
    verbose_eval=100,
)
es_fired = booster_supp.best_iteration < (2000 - 50)
print(f'best iteration: {booster_supp.best_iteration} of 2000 · early stopping fired: {es_fired}')

[0]	train-aucpr:0.00792	val-aucpr:0.00609
[100]	train-aucpr:0.14986	val-aucpr:0.05542
[200]	train-aucpr:0.21762	val-aucpr:0.06585
[300]	train-aucpr:0.26848	val-aucpr:0.07327
[400]	train-aucpr:0.31718	val-aucpr:0.07711
[479]	train-aucpr:0.34897	val-aucpr:0.07875
best iteration: 429 of 2000 · early stopping fired: True


In [21]:
# same seed and params mean the early rounds are identical; this checks it
# rather than assuming it
p_supp = booster_supp.predict(dmain, iteration_range=(0, booster_supp.best_iteration + 1))
supp_pr = average_precision_score(y_main, p_supp)

print(f'best_iteration — registered {booster.best_iteration} · supplementary {booster_supp.best_iteration}')
print(f'max abs prob difference on test-main: {np.abs(p_supp - p_main).max():.3e}')
print(f'test-main PR-AUC — registered {m_main["pr_auc"]:.6f} · supplementary {supp_pr:.6f}')

json.dump({'registered_best_iteration': int(booster.best_iteration),
           'supp_best_iteration': int(booster_supp.best_iteration),
           'supp_early_stopping_fired': bool(es_fired),
           'max_abs_prob_diff': float(np.abs(p_supp - p_main).max()),
           'registered_pr_auc': float(m_main['pr_auc']),
           'supp_pr_auc': float(supp_pr)},
          open(OUT_DIR / 'models' / '07_m2_supplementary.json', 'w'), indent=2)

best_iteration — registered 429 · supplementary 429
max abs prob difference on test-main: 0.000e+00
test-main PR-AUC — registered 0.063708 · supplementary 0.063708
